# InternVideo2-CLIP: MSVD + MSR-VTT Fine-Tuning

This notebook fine-tunes the InternVideo2-CLIP dual-encoder model on the MSVD training split (1,200 videos) and MSR-VTT training split (7,010 videos) for 10 epochs using cross-modal contrastive InfoNCE loss.


In [ ]:
!pip -q install transformers onnxruntime-gpu


In [ ]:
from pathlib import Path
import os, shutil, zipfile

ROOT = Path("/content/data")
required = {
    "msvd_train.json": "msvd", "msvd_test.json": "msvd",
    "msrvtt_train_7k.json": "msrvtt", "msrvtt_test_1k.json": "msrvtt",
}
for dataset in ("msvd", "msrvtt"):
    (ROOT / dataset / "videos").mkdir(parents=True, exist_ok=True)

# Find uploaded annotation JSON files.
found = {}
for folder, dirs, names in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in {".cache", "sample_data", "checkpoints", "tokenizer", "videos", "YouTubeClips"}]
    for name in required.keys() & set(names):
        found.setdefault(name, Path(folder) / name)
missing = required.keys() - found.keys()
if missing:
    raise FileNotFoundError(f"Missing uploaded annotation files: {sorted(missing)}")
for name, dataset in required.items():
    target = ROOT / dataset / name
    source = found[name]
    if source.resolve() != target.resolve():
        shutil.copy2(source, target)

# Read the exact video names required by the four annotation files.
import json
wanted = {"msvd": set(), "msrvtt": set()}
for name, dataset in required.items():
    for row in json.loads((ROOT / dataset / name).read_text()):
        filename = row.get("video") or (row["video_id"] + (".avi" if dataset == "msvd" else ".mp4"))
        wanted[dataset].add(Path(filename).name)

# Extract uploaded archives by video type; nested folder layouts are flattened.
archives = []
for folder, dirs, names in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in {".cache", "sample_data", "checkpoints", "tokenizer"}]
    archives.extend(Path(folder) / n for n in names if n.lower().endswith(".zip"))
print("ZIP files found:", [(str(p), round(p.stat().st_size / 1e9, 2)) for p in archives])
for archive_path in archives:
    try:
        with zipfile.ZipFile(archive_path) as archive:
            members = archive.namelist()
            if any(m.lower().endswith(".avi") for m in members): dataset, ext = "msvd", ".avi"
            elif any(m.lower().endswith(".mp4") for m in members): dataset, ext = "msrvtt", ".mp4"
            else: continue
            existing = {p.name for p in (ROOT / dataset / "videos").rglob(f"*{ext}")}
            members = [m for m in members if m.lower().endswith(ext)]
            print(f"{archive_path.name}: {len(members)} {dataset} video entries")
            if not wanted[dataset].issubset(existing):
                archive.extractall(ROOT / dataset / "videos")
    except zipfile.BadZipFile:
        print("Skipping invalid ZIP:", archive_path)

# Link individually uploaded video files into the expected dataset folders.
for folder, dirs, names in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in {".cache", "sample_data", "checkpoints", "tokenizer"}]
    for dataset, ext in (("msvd", ".avi"), ("msrvtt", ".mp4")):
        for name in set(names) & wanted[dataset]:
            source = Path(folder) / name
            target = ROOT / dataset / "videos" / name
            if not target.exists(): target.symlink_to(source)

for dataset, ext in (("msvd", ".avi"), ("msrvtt", ".mp4")):
    video_dir = ROOT / dataset / "videos"
    for source in list(video_dir.rglob(f"*{ext}")):
        target = video_dir / source.name
        if source != target and not target.exists(): shutil.move(str(source), str(target))
    missing_videos = sorted(name for name in wanted[dataset] if not (video_dir / name).is_file())
    if dataset == "msrvtt" and missing_videos:
        import urllib.request
        full_archive = Path("/content/MSRVTT_Videos.zip")
        if not full_archive.exists():
            print("The uploaded files contain only part of MSR-VTT. Downloading the full 2.19 GB video archive...")
            urllib.request.urlretrieve(
                "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/MSRVTT_Videos.zip",
                full_archive,
            )
        with zipfile.ZipFile(full_archive) as archive:
            archive.extractall(video_dir)
        for source in list(video_dir.rglob("*.mp4")):
            target = video_dir / source.name
            if source != target and not target.exists(): shutil.move(str(source), str(target))
        missing_videos = sorted(name for name in wanted[dataset] if not (video_dir / name).is_file())
    print(f"{dataset}: {len(wanted[dataset]) - len(missing_videos)}/{len(wanted[dataset])} required videos found")
    if missing_videos:
        raise FileNotFoundError(f"{dataset} is missing {len(missing_videos)} videos, e.g. {missing_videos[:8]}")
print("Both datasets are staged. Continue to the training cell.")


## Cross-Modal Retrieval Adapters and Contrastive Optimization

This stage defines the feature encoders, projection adapters, cached feature datasets, and the symmetric contrastive training loop. Checkpoints are saved per epoch to `/content/checkpoints/internvideo2_clip_finetuned`.


In [ ]:
import os
import json
import random
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import onnxruntime as ort
from transformers import AutoTokenizer

ROOT = Path(/content/data)
OUTPUT_DIR = Path(/content/checkpoints/internvideo2_clip_finetuned)
DATASETS = (msvd, msrvtt)


def load_encoder():
    providers = [CUDAExecutionProvider, CPUExecutionProvider] if torch.cuda.is_available() else [CPUExecutionProvider]
    vision = ort.InferenceSession(/content/iv2clip_vision_fp16.onnx, providers=providers)
    text = ort.InferenceSession(/content/iv2clip_text_fp16.onnx, providers=providers)
    tokenizer = AutoTokenizer.from_pretrained(/content/tokenizer)
    return vision, text, tokenizer


class Encoder:
    def __init__(self, vision_sess, text_sess, tokenizer):
        self.vision_sess = vision_sess
        self.text_sess = text_sess
        self.tokenizer = tokenizer
        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def extract_frames(self, video_path, num_frames=8):
        cap = cv2.VideoCapture(str(video_path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        idxs = np.linspace(0, total - 1, num_frames, dtype=int)
        frames = []
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret:
                frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        cap.release()
        while len(frames) < num_frames:
            frames.append(frames[-1] if frames else Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)))
        return torch.stack([self.transform(f) for f in frames[:num_frames]]).numpy()

    def encode_video(self, video_path):
        frames = self.extract_frames(video_path, num_frames=8)
        batch = np.expand_dims(frames, axis=0).astype(np.float32)
        out = self.vision_sess.run([image_embedding], {image: batch})[0]
        norm = out / np.linalg.norm(out, axis=-1, keepdims=True)
        return norm[0]

    def encode_texts(self, captions, batch_size=64):
        all_embs = []
        for i in range(0, len(captions), batch_size):
            batch = captions[i:i+batch_size]
            enc = self.tokenizer(batch, padding=max_length, max_length=77, truncation=True, return_tensors=np)
            out = self.text_sess.run([text_embedding], {
                input_ids: enc[input_ids].astype(np.int64),
                attention_mask: enc[attention_mask].astype(np.int64)
            })[0]
            norm = out / np.linalg.norm(out, axis=-1, keepdims=True)
            all_embs.append(norm)
        return np.concatenate(all_embs, axis=0)


def read_items(dataset, split=train):
    suffix = "_7k" if dataset == "msrvtt" and split == "train" else "_1k" if dataset == "msrvtt" and split == "test" else ""
    ann_file = ROOT / dataset / f"{dataset}_{split}{suffix}.json"
    rows = json.loads(ann_file.read_text())
    items = []
    video_dir = ROOT / dataset / "videos"
    for r in rows:
        fn = r.get("video") or (r["video_id"] + (".avi" if dataset == "msvd" else ".mp4"))
        vpath = video_dir / Path(fn).name
        if vpath.is_file():
            caption = r["caption"]
            if isinstance(caption, list):
                caption = caption[0]
            items.append({"video": str(vpath), "caption": caption, "video_id": Path(fn).stem})
    return items


def feature_cache(dataset, split, items, encoder):
    cache_dir = Path("/content/cache") / dataset
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{split}_video_features.npy"
    if cache_path.exists():
        return np.load(cache_path, allow_pickle=True).item()
    feats = {}
    unique_videos = list({it["video"] for it in items})
    for v in tqdm(unique_videos, desc=f"Caching {dataset} videos"):
        feats[v] = encoder.encode_video(v)
    np.save(cache_path, feats)
    return feats


def caption_features(items, encoder, train=True):
    captions = [it["caption"] for it in items]
    return encoder.encode_texts(captions)


class VideoPairs(Dataset):
    def __init__(self, items, video_features, text_features, seed=42):
        self.items = items
        self.video_features = video_features
        self.text_features = text_features

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        v_feat = torch.from_numpy(self.video_features[item["video"]]).float()
        t_feat = torch.from_numpy(self.text_features[idx]).float()
        return v_feat, t_feat


class Adapter(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )

    def forward(self, x):
        return x + 0.1 * self.net(x)


class RetrievalAdapters(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.video_adapter = Adapter(dim)
        self.text_adapter = Adapter(dim)
        self.logit_scale = nn.Parameter(torch.ones([]) * float(np.log(1 / 0.07)))

    def forward(self, v, t):
        v = F.normalize(self.video_adapter(v), dim=-1)
        t = F.normalize(self.text_adapter(t), dim=-1)
        return v, t


def symmetric_loss(model, video, caption):
    v, t = model(video, caption)
    logit_scale = model.logit_scale.exp()
    logits_per_video = logit_scale * (v @ t.T)
    logits_per_text = logits_per_video.T
    labels = torch.arange(len(logits_per_video), device=video.device)
    loss = (F.cross_entropy(logits_per_video, labels) + F.cross_entropy(logits_per_text, labels)) / 2
    return loss



def run(epochs=10, batch_size=64, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

    vision, text, tokenizer = load_encoder()
    encoder = Encoder(vision, text, tokenizer)

    train_rows, video_features, text_features = {}, {}, {}
    for dataset in DATASETS:
        train_rows[dataset] = read_items(dataset, "train")
        video_features[dataset] = feature_cache(
            dataset, "train", train_rows[dataset], encoder
        )
        text_features[dataset] = caption_features(
            train_rows[dataset], encoder, train=True
        )
        print(f"{dataset}: {len(train_rows[dataset])} training caption rows")

    train_sets = {
        dataset: VideoPairs(
            train_rows[dataset],
            video_features[dataset],
            text_features[dataset],
            seed,
        )
        for dataset in DATASETS
    }
    loaders = {
        dataset: DataLoader(
            train_sets[dataset],
            batch_size=batch_size,
            shuffle=True,
            drop_last=True,
        )
        for dataset in DATASETS
    }
    if any(len(loader) == 0 for loader in loaders.values()):
        raise RuntimeError(
            "Each dataset needs at least one complete training batch of matched videos"
        )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for old_checkpoint in OUTPUT_DIR.glob("video2clip_epoch_*.pth"):
        old_checkpoint.unlink()
    for old_name in ("best_video2clip_adapters.pth", "retrieval_results.json"):
        old_file = OUTPUT_DIR / old_name
        if old_file.exists():
            old_file.unlink()

    model = RetrievalAdapters().to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=5e-5,
        weight_decay=1e-3,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=2e-5,
    )

    baseline_path = OUTPUT_DIR / "video2clip_epoch_00_baseline.pth"
    torch.save(
        {"epoch": 0, "adapter_state_dict": model.state_dict()},
        baseline_path,
    )

    for epoch in range(1, epochs + 1):
        model.train()
        iterators = {
            dataset: iter(loaders[dataset])
            for dataset in DATASETS
        }
        steps = min(len(loaders[dataset]) for dataset in DATASETS)
        losses = []

        for _ in tqdm(range(steps), desc=f"Epoch {epoch}/{epochs}"):
            optimizer.zero_grad(set_to_none=True)
            domain_losses = []

            for dataset in DATASETS:
                video, caption = next(iterators[dataset])
                domain_losses.append(
                    symmetric_loss(
                        model,
                        video.to(device),
                        caption.to(device),
                    )
                )

            loss = torch.stack(domain_losses).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            with torch.no_grad():
                model.logit_scale.clamp_(max=float(np.log(100)))

            losses.append(loss.item())

        scheduler.step()
        checkpoint_path = OUTPUT_DIR / f"video2clip_epoch_{epoch:02d}.pth"
        torch.save(
            {
                "epoch": epoch,
                "train_loss": float(np.mean(losses)),
                "adapter_state_dict": model.state_dict(),
            },
            checkpoint_path,
        )
        print(
            f"epoch={epoch} loss={np.mean(losses):.4f}; "
            f"saved {checkpoint_path.name}"
        )

    print(f"Training finished. Checkpoints are in {OUTPUT_DIR}.")
    print("Run the separate evaluation cell after training to compare them.")


if __name__ == "__main__":
    run()